# 10 - Gols por Minuto e Cartões vs Posição

- Distribuição de gols por minuto — tem pico após o intervalo?
- Cartões vs posição na tabela — times desesperados jogam mais sujo?
- Times que fazem o primeiro gol vencem em quantos % dos casos?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)
# Fix temporada 2020 (COVID): rodadas 28-38 jogadas em jan-fev/2021 pertencem a temporada 2020
mask_2020 = (df['data'].dt.year == 2021) & (df['rodata'] >= 28) & (df['data'] < '2021-03-01')
df.loc[mask_2020, 'ano'] = 2020
df = df[df['ano'] >= 2003].copy()

df_gols = pd.read_csv('../data/raw/campeonato-brasileiro-gols.csv')
df_cartoes = pd.read_csv('../data/raw/campeonato-brasileiro-cartoes.csv')
df_stats = pd.read_csv('../data/raw/campeonato-brasileiro-estatisticas-full.csv')

print(f'Partidas: {len(df)}')
print(f'Gols: {len(df_gols)}')
print(f'Cart\u00f5es: {len(df_cartoes)}')
print(f'Estat\u00edsticas: {len(df_stats)}')

Partidas: 9225
Gols: 9861
Cartões: 20953
Estatísticas: 17570


In [2]:
# Limpar minuto dos gols (remover '+' de acréscimos e converter)
def parse_minuto(m):
    if pd.isna(m):
        return None
    m = str(m).strip()
    if '+' in m:
        parts = m.split('+')
        try:
            return int(parts[0]) + int(parts[1])
        except:
            return int(parts[0])
    try:
        return int(m.replace("'", '').strip())
    except:
        return None

df_gols['minuto_num'] = df_gols['minuto'].apply(parse_minuto)
df_gols_clean = df_gols.dropna(subset=['minuto_num'])
df_gols_clean['minuto_num'] = df_gols_clean['minuto_num'].astype(int)

# Limitar a 0-100 (gols razoáveis)
df_gols_clean = df_gols_clean[(df_gols_clean['minuto_num'] >= 0) & (df_gols_clean['minuto_num'] <= 100)]
print(f'Gols com minuto v\u00e1lido: {len(df_gols_clean)}')

Gols com minuto válido: 9847


In [3]:
# Distribuição de gols por minuto
gols_por_min = df_gols_clean.groupby('minuto_num').size().reset_index(name='gols')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=gols_por_min['minuto_num'],
    y=gols_por_min['gols'],
    marker_color=['#e74c3c' if m > 45 and m <= 55 else '#3498db' if m > 75 else '#95a5a6'
                  for m in gols_por_min['minuto_num']],
    hovertemplate="Minuto %{x}': %{y} gols<extra></extra>"
))

# Linha do intervalo
fig.add_vline(x=45, line_dash='dash', line_color='black', opacity=0.5,
              annotation_text='Intervalo', annotation_position='top')

fig.update_layout(
    title='Distribuição de Gols por Minuto do Jogo<br><sup>Brasileirão Série A (2003-presente) | Vermelho = pós-intervalo | Azul = minutos finais</sup>',
    xaxis_title='Minuto',
    yaxis_title='Número de Gols',
    template='plotly_white',
    width=950,
    height=500
)

fig.show()
_path = '../charts/gols_por_minuto.html'
fig.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Distribuição temporal dos gols ao longo dos 90+ minutos: observa-se tendência crescente na frequência de gols após o intervalo, com pico nos minutos finais (75'-90')."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/gols_por_minuto.html')

Gráfico exportado: charts/gols_por_minuto.html


In [4]:
# Gols por faixa de tempo (15 min)
def faixa_tempo(m):
    if m <= 15: return '0-15'
    elif m <= 30: return '16-30'
    elif m <= 45: return '31-45'
    elif m <= 60: return '46-60'
    elif m <= 75: return '61-75'
    else: return '76-90+'

df_gols_clean = df_gols_clean.copy()
df_gols_clean['faixa'] = df_gols_clean['minuto_num'].apply(faixa_tempo)

faixas = df_gols_clean.groupby('faixa').size().reset_index(name='gols')
faixas['faixa'] = pd.Categorical(faixas['faixa'], ['0-15','16-30','31-45','46-60','61-75','76-90+'])
faixas = faixas.sort_values('faixa')
faixas['pct'] = faixas['gols'] / faixas['gols'].sum() * 100

print('Gols por faixa de tempo:')
for _, r in faixas.iterrows():
    print(f"  {r['faixa']}': {r['gols']} gols ({r['pct']:.1f}%)")

print(f"\n1\u00ba tempo: {faixas[faixas['faixa'].isin(['0-15','16-30','31-45'])]['gols'].sum()} gols")
print(f"2\u00ba tempo: {faixas[faixas['faixa'].isin(['46-60','61-75','76-90+'])]['gols'].sum()} gols")

Gols por faixa de tempo:
  0-15': 1201 gols (12.2%)
  16-30': 1330 gols (13.5%)
  31-45': 1495 gols (15.2%)
  46-60': 1891 gols (19.2%)
  61-75': 1623 gols (16.5%)
  76-90+': 2307 gols (23.4%)

1º tempo: 4026 gols
2º tempo: 5821 gols


In [5]:
# PRIMEIRO GOL DECIDE?
# Para cada partida, ver quem fez o 1º gol e se venceu
df_gols_ord = df_gols_clean.sort_values(['partida_id', 'minuto_num'])
primeiro_gol = df_gols_ord.groupby('partida_id').first().reset_index()

# Juntar com resultados das partidas
df_merged = primeiro_gol[['partida_id', 'clube']].merge(
    df[['ID', 'mandante', 'visitante', 'vencedor']].rename(columns={'ID': 'partida_id'}),
    on='partida_id', how='inner'
)

df_merged['fez_1o_gol_venceu'] = df_merged['clube'] == df_merged['vencedor']
df_merged['empate'] = df_merged['vencedor'] == '-'

total = len(df_merged)
venceu = df_merged['fez_1o_gol_venceu'].sum()
empatou = df_merged['empate'].sum()
perdeu = total - venceu - empatou

print(f'Quem faz o 1º gol...')
print(f'  Vence: {venceu} ({venceu/total*100:.1f}%)')
print(f'  Empata: {empatou} ({empatou/total*100:.1f}%)')
print(f'  Perde: {perdeu} ({perdeu/total*100:.1f}%)')

# Gráfico de pizza
fig2 = go.Figure(data=[go.Pie(
    labels=['Vence', 'Empata', 'Perde'],
    values=[venceu, empatou, perdeu],
    marker_colors=['#27ae60', '#f39c12', '#e74c3c'],
    hole=0.4,
    textinfo='label+percent',
    hovertemplate='%{label}: %{value} partidas (%{percent})<extra></extra>'
)])

fig2.update_layout(
    title='Quem Faz o Primeiro Gol...<br><sup>Brasileirão Série A (2003-presente)</sup>',
    template='plotly_white',
    width=600,
    height=500
)

fig2.show()
_path = '../charts/primeiro_gol.html'
fig2.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Proporção de resultados condicionada ao primeiro gol marcado: a probabilidade condicional de vitória é superior a 70%, evidenciando forte correlação entre abrir o placar e vencer a partida."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/primeiro_gol.html')

Quem faz o 1º gol...
  Vence: 2676 (70.1%)
  Empata: 756 (19.8%)
  Perde: 386 (10.1%)


Gráfico exportado: charts/primeiro_gol.html


In [6]:
# CARTÕES vs POSIÇÃO NA TABELA
# Calcular classificação final e total de cartões por time por temporada
def classificacao_final(df_season):
    teams = set(df_season['mandante'].unique()) | set(df_season['visitante'].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    for _, jogo in df_season.iterrows():
        m, v = jogo['mandante'], jogo['visitante']
        gm, gv = jogo['mandante_Placar'], jogo['visitante_Placar']
        if pd.isna(gm) or pd.isna(gv): continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv; saldo[v] += gv - gm
        if gm > gv: pontos[m] += 3; vitorias[m] += 1
        elif gm < gv: pontos[v] += 3; vitorias[v] += 1
        else: pontos[m] += 1; pontos[v] += 1
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return {t: i+1 for i, t in enumerate(ranking)}

# Juntar partida_id com ano
df_id_ano = df[['ID', 'ano']].rename(columns={'ID': 'partida_id'})
df_cartoes_ano = df_cartoes.merge(df_id_ano, on='partida_id', how='inner')

cartoes_time = []
for ano in sorted(df['ano'].unique()):
    df_s = df[df['ano'] == ano]
    df_s_clean = df_s.dropna(subset=['rodata'])
    if df_s_clean['rodata'].astype(int).max() < 30: continue
    classif = classificacao_final(df_s)
    
    c_ano = df_cartoes_ano[df_cartoes_ano['ano'] == ano]
    amarelos = c_ano[c_ano['cartao'] == 'Amarelo'].groupby('clube').size()
    vermelhos = c_ano[c_ano['cartao'] == 'Vermelho'].groupby('clube').size()
    
    for time, pos in classif.items():
        cartoes_time.append({
            'ano': ano, 'time': time, 'posicao': pos,
            'amarelos': amarelos.get(time, 0),
            'vermelhos': vermelhos.get(time, 0),
            'total_cartoes': amarelos.get(time, 0) + vermelhos.get(time, 0)
        })

df_cartoes_pos = pd.DataFrame(cartoes_time)

# Agrupar por faixa de posição
def faixa_pos(p):
    if p <= 4: return 'Top 4'
    elif p <= 10: return '5\u00ba-10\u00ba'
    elif p <= 16: return '11\u00ba-16\u00ba'
    else: return 'Z4 (17\u00ba-20\u00ba)'

df_cartoes_pos['faixa'] = df_cartoes_pos['posicao'].apply(faixa_pos)
df_cartoes_pos['faixa'] = pd.Categorical(df_cartoes_pos['faixa'], ['Top 4', '5\u00ba-10\u00ba', '11\u00ba-16\u00ba', 'Z4 (17\u00ba-20\u00ba)'])

print('M\u00e9dia de cart\u00f5es por faixa de classifica\u00e7\u00e3o:')
for faixa in ['Top 4', '5\u00ba-10\u00ba', '11\u00ba-16\u00ba', 'Z4 (17\u00ba-20\u00ba)']:
    sub = df_cartoes_pos[df_cartoes_pos['faixa'] == faixa]
    print(f"  {faixa}: {sub['amarelos'].mean():.1f} amarelos, {sub['vermelhos'].mean():.1f} vermelhos")

Média de cartões por faixa de classificação:
  Top 4: 40.8 amarelos, 2.1 vermelhos
  5º-10º: 42.6 amarelos, 1.9 vermelhos
  11º-16º: 43.8 amarelos, 2.6 vermelhos
  Z4 (17º-20º): 40.8 amarelos, 2.7 vermelhos


In [7]:
# Scatter: posição vs cartões
corr = df_cartoes_pos['posicao'].corr(df_cartoes_pos['total_cartoes'])

fig3 = px.scatter(df_cartoes_pos, x='posicao', y='total_cartoes',
                  color='faixa',
                  color_discrete_map={'Top 4': '#27ae60', '5\u00ba-10\u00ba': '#3498db',
                                      '11\u00ba-16\u00ba': '#f39c12', 'Z4 (17\u00ba-20\u00ba)': '#e74c3c'},
                  opacity=0.5,
                  hover_data=['time', 'ano'],
                  title=f'Posi\u00e7\u00e3o na Tabela vs Cart\u00f5es Recebidos<br><sup>Correla\u00e7\u00e3o: r = {corr:.3f} | Brasileir\u00e3o (2003-presente)</sup>',
                  labels={'posicao': 'Posi\u00e7\u00e3o Final', 'total_cartoes': 'Total de Cart\u00f5es', 'faixa': 'Faixa'})

# Linha de tendência
z = np.polyfit(df_cartoes_pos['posicao'], df_cartoes_pos['total_cartoes'], 1)
x_line = np.linspace(1, 24, 100)
fig3.add_trace(go.Scatter(x=x_line, y=np.polyval(z, x_line),
                          mode='lines', line=dict(color='gray', dash='dash'),
                          name='Tend\u00eancia', showlegend=False))

fig3.update_layout(template='plotly_white', width=800, height=550)

fig3.show()
_path = '../charts/cartoes_posicao.html'
fig3.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Dispersão entre posição final e total de cartões com correlação praticamente nula (r \u2248 -0.018), indicando ausência de associação linear significativa entre indisciplina e desempenho na tabela."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/cartoes_posicao.html')

Gráfico exportado: charts/cartoes_posicao.html


In [8]:
# Heatmap: gols por minuto x rodada
df_gols_rod = df_gols_clean.copy()
df_gols_rod = df_gols_rod.dropna(subset=['rodata'])
df_gols_rod['rodata'] = df_gols_rod['rodata'].astype(int)
df_gols_rod = df_gols_rod[df_gols_rod['rodata'] <= 38]
df_gols_rod['faixa_min'] = (df_gols_rod['minuto_num'] // 15) * 15
df_gols_rod['faixa_min_label'] = df_gols_rod['faixa_min'].apply(lambda x: f"{x}-{x+14}'")

heat = df_gols_rod.groupby(['faixa_min_label', 'rodata']).size().reset_index(name='gols')
heat_pivot = heat.pivot(index='faixa_min_label', columns='rodata', values='gols').fillna(0)

# Reordenar faixas
order = ["0-14'", "15-29'", "30-44'", "45-59'", "60-74'", "75-89'"]
heat_pivot = heat_pivot.reindex([f for f in order if f in heat_pivot.index])

fig4 = px.imshow(heat_pivot, 
                 labels=dict(x='Rodada', y='Faixa de Minuto', color='Gols'),
                 color_continuous_scale='YlOrRd',
                 title='Heatmap: Gols por Faixa de Minuto x Rodada<br><sup>Brasileirão Série A (2003-presente)</sup>',
                 aspect='auto')
fig4.update_layout(template='plotly_white', width=1000, height=400)

fig4.show()
_path = '../charts/heatmap_gols_minuto_rodada.html'
fig4.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Heatmap bidimensional de gols por faixa de minuto e rodada: as faixas 75-89' e 45-59' concentram as maiores frequências absolutas."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/heatmap_gols_minuto_rodada.html')

Gráfico exportado: charts/heatmap_gols_minuto_rodada.html
